## What is a Data Warehouse 

**Why**

- Analysts who run queries aggregating millions of rows can bring down the production database.

- Analysts are unable to join data across your company’s multiple databases.
- Data users require historical information to track and forecast performance metrics.

**What**

- Data warehousing is a process of getting data ready for analytics.

- Importantly, this process is independent of the database type. Application databases are meant for CRUD operations on a few rows at a time, whereas analytical databases are designed to aggregate data over millions of rows.

**How**

- Data engineers play a key role in this process by collaborating across systems and teams to create analytics-ready data sets.
- We'll explore each part of this process in detail throughout the course.

![Analytical Query](images/analytical_query.png)

#### Example

Analytical question: Which product variants are driving the most sales?

In [ ]:
%%sql
SELECT
    p.name                            AS product_name,
    pv.sku,
    pv.size,
    pv.color,
    COUNT(DISTINCT ol.order_id)       AS orders,
    SUM(ol.quantity)                  AS units_sold,
    ROUND(SUM(ol.line_total), 2)      AS total_revenue
FROM order_line ol
JOIN product_variant pv ON pv.variant_id  = ol.variant_id
JOIN product         p  ON p.product_id   = pv.product_id
GROUP BY 1, 2, 3, 4
ORDER BY total_revenue DESC
LIMIT 4;

**Tradeoffs**

- Building a warehouse is a lot of work and ongoing maintenance
- In startup/small companies, most use cases are solvable using the application tables directly (or using a read-only copy)

#### Exercise [10 min]

Consider that your company has two microservices. 

![Basic Data Warehouse](images/basic_data_warehouse.png)

Your stakeholders need to track metrics that involve data from both the microservice's database. Think about the following scenarios.

1. What challenges arise when analysts query & use data from both microservice's databases?
2. What should you ask stakeholders before deciding to build a warehouse?
3. What simpler alternatives to a warehouse could you try?

## Kimball Data Model 

- After consolidating data from multiple sources into a single system, we face a new challenge: making it accessible and useful for analytics. This transition is where data modeling becomes essential.

- Modeling data = table schemas that makes data more accessible to stakeholders.

- Without data modeling, your analysts write inconsistent queries, metric definitions diverge, and every report becomes a one-off.

- There are multiple established ways to model data for warehousing

- The most common and popular one is called Kimball Data Modeling (aka dimensional data model)

- In Kimball data modeling, you designate tables as one of: Facts, Dimensions, or Bridge tables.

#### Example
The key idea is a star schema using fact, dimension, and bridge tables.

![Star Schema](images/star_schema_orders_example.png)

A star schema features a fact table surrounded by dimension tables, forming a star pattern.

- In the Kimball data model, there are 3 main types of tables
  - **Dimension**: A table representing a business entity. e.g., User, Customer, Seller, Some attributes of an entity can also have their own dimension. Eg. user_credit_cards, etc
  - **Facts**: A table in which each row represents a real-world event that happened, e.g., an order from a website. Note that facts typically happen at the intersection of dimensions, e.g., a seller sells a product to a customer. This event has 3 dimensions: Seller, Product, and Customer.
  - **Bridge**: There are dimensions that are related to each other in a many-to-many relationship. In such cases, we use a bridge table to map dim1_id and dim2_id. e.g., "a product can belong to many promotions, and a promotion can apply to many products." You can't store this relationship cleanly on either dimension table without duplication, so a bridge table maps the two (dim_product and dim_promotion) and have the schema `product_id,promotion_id`.

#### Exercise [5 min]

Based on the shown image below, what will the facts and dimensions be for our tables?
![Star Schema](images/star_schema_orders_exercise.png)

## Analytical queries involve joining fact and dimension tables and grouping by dimension attribute(s)

#### Example

- Let's consider a simple analytical question: what are the top 5 product variants (in terms of their revenue) over the last 2 years, and show me the number of orders that they were a part of, the number of units sold, and the total revenue they generated.



In [ ]:
%%sql
SELECT
    -- dimensions
    ol.created_at::DATE               AS order_date,
    p.name                            AS product_name,
    pv.sku,
    pv.size,
    pv.color,

    -- metrics
    COUNT(DISTINCT ol.order_id)       AS orders,
    SUM(ol.quantity)                  AS units_sold,
    ROUND(SUM(ol.line_total), 2)      AS total_revenue
FROM order_line ol                                                    -- fact
LEFT JOIN product_variant pv ON pv.variant_id  = ol.variant_id             -- dimension
LEFT JOIN product         p  ON p.product_id   = pv.product_id             -- dimension
WHERE ol.created_at >= NOW() - INTERVAL '2 years'                     
GROUP BY 1, 2, 3, 4, 5
ORDER BY total_revenue DESC
LIMIT 5

- From the above query, we can see how we join the dimension-type tables `product` and `product_variant` with the fact `order_line` table, and then we group by the dimensional attribute (variant ID and details)
- Could `product` and `product_variant` be a single dimension?
- This approach forms a common pattern for most analytical querying tasks.
- We can see how creating fact and dimension tables helps write analytical queries
- The calculated field, often called metrics, is an aggregate of numerical data (or count of IDs) from the fact table. Since we are analyzing history, the fact table represents historical data.
- As data engineers, we will need to make sure that when we create the fact and dimension tables, the joins work as expected, data types are accurate, data is correct, etc

#### Exercise [10 min]
For every month (in "order" table), show me the unique number of customers, orders, & total_revenue. The results should be ordered by the month in ascending order.

*Hint*: Use `DATE_TRUNC('month', date_col)` to get the month. 

Use `select * from "order"` format to access data from the order table.

## Facts are generated by your system, the user's browser, or purchased from a third-party

- Facts represent real-life events and can be generated by

  - Your user's browser (aka user client), which logs their interaction with our website (e.g., impression, click, etc.). Other examples are systems like client server logs, any client system that generates data
  - Most financial transactions occur on company servers, which capture them. E.g., a buyer purchasing an item from a seller on an e-commerce website
  - 3rd-party data providers work with companies to provide data that the company may not have access to otherwise. 

- When capturing fact data, we acquire all the information we can. For example: user interaction on a website will include time of interaction, browser name and version (aka user agent), cookie ID, etc

- Fact-generating code tries to associate a fact with all the possible dimension IDs
  - E.g., A user interaction will capture the user_id (if they are logged in). 
  - E.g. A financial transaction captures user_id and product_id(s). 

- Fact data is typically loaded into an append-only system, e.g., an append-only DB table, a Kafka log dumping data into S3 buckets, etc. 

- Fact data is almost never deleted.

- **Grain = what one row in your table represents. This is also known as the table's level.**
- 1 table = 1 grain. Do not break this rule, unless absolutely necessary

#### Example

- Both order_line & order are fact tables with different grains.
- 1 row of order_line =  1 item purchased as part of an order  

- 1 row of the order table = 1 entire order

In [ ]:
%%sql
select * from "order" order by order_id limit 1

In [ ]:
%%sql
select * from order_line where order_id in (select order_id from "order" order by order_id limit 1)

* Caution: tables with multiple grains are almost always a bad idea and will cause a lot of confusion for downstream users

#### Exercise [5 min]

Which of the following tables are fact tables?


In [ ]:
%%sql
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public' 
  AND table_type = 'BASE TABLE'
AND table_name in ('ad_click', 'product', 'ad_impression', 'campaign', 'ad_group')

## Dimensions represent a business concept

- The business concept that a dimension table represents can be
  - A user of the business: customer, seller, supplier, delivery_person, etc
  - A feature of the business: credit_card_detail, ad_campaign, promotion details, etc
  - Almost everything besides a fact table

- Dimensions represent the interacting parts of a fact table 

- Dimension tables are formed by combining multiple source datasets (tables, API, etc.)

- The grain of a dimension table should be the "thing" that it represents. E.g., The grain of a user dimension table  will be a user

#### Example

Consider these source system tables 
- product_category
- product
- product_variant
- product_attribute

- All of the tables above represent product data. We can ingest them into our warehouse or leave them as is. But that would make data use difficult and error-prone.

```text
product_category                     # 1: Many     
  └── product                        # 1: Many
        └── product_variant          # 1: Many
              └── product_attribute  # 1: Many
```

- We can create a dim_product_variant table with data from the following tables

```text
dim_product_variant
-- data from product_variant
-- data from product
-- data from product_category
```

- **Note** A product has one category and may have multiple variants. In `order_line`, the items sold are `product_variants`.

- We keep the grain at the product_variant level, as this is how they are sold and tracked in the fact table.

- A product variant can have multiple attributes (1:many). Given this, we can either keep product_attributes as a separate dimension or add them as an array[struct] column to our dim_product_variant table. We will cover this in the data model chapter.

- According to Kimball's dimensional modelling, there are 7 ways to model a dimension table. 

- But with advances in technology and a reduction in storage costs, most companies use the following 2-dimensional types
    - **Snapshot (also known as SCD1) dimensions:** With each pipeline run, all source data is fully reprocessed and the dimension table is replaced with updated data.
    - **Slowly Changing Dimension (SCD2):** This approach records a changelog reflecting every alteration to the represented entity. Each row includes `valid_from` and `valid_to` timestamps that indicate the period during which that row describes the entity’s state. They also include `is_current`, a boolean column indicating if the row is the most current state of the data.

- Snapshots are much easier to build and maintain compared to SCD2. 

- Use SCD2 only when you absolutely need to track every change to the dimension.

#### Exercise [10 min]

Consider this upstream table called `inventory`. What would the SCD2 version of this table look like in your warehouse?

We see what the `inventory` data looks like over 3 consecutive days and assume you have a pipeline that processes it once a day.

**Day 1**

Input:
```markdown
| id | qty_on_hand | updated_at |
|----|-------------|------------|
| 1  | 100         | day1       |
```

---

**Day 2** 

Input:
```markdown
| id | qty_on_hand | updated_at |
|----|-------------|------------|
| 1  | 100         | day1       |
```

---

**Day 3** 

Input:
```markdown
| id | qty_on_hand | updated_at |
|----|-------------|------------|
| 1  | 70          | day3       |
```

* In snapshot dimensions you can use the id as the unique key, however this will not work in SCD2 since the id will be repeated. 

* In warehouse **natural** key is used to refer to the primary key of the source tables.
* Kimball's recommendation is to include a surrogate key (e.g. an increasing id) to be able to uniquely identify each row. This would then be used to enrich fact tables, making joins easy.
* With faster compute, we can join the fact and SCD2 tables directly using id and time range (matching fact created time to valid_from and valid_to), instead of relying on a surrogate key.

#### Exercise [10 min]

Given below are sample facts and SCD2 tables. Write a sql query to join them as we discussed

In [ ]:
%%sql
-- ============================================================
-- TEMP TABLES: dim_customer (SCD2) and fct_order
-- ============================================================

CREATE TEMP TABLE dim_customer (
    customer_key    SERIAL PRIMARY KEY,   -- surrogate key
    customer_id     INT,                  -- natural key
    full_name       TEXT,
    status          TEXT,
    effective_from  TIMESTAMP,
    effective_to    TIMESTAMP
);

INSERT INTO dim_customer (customer_id, full_name, status, effective_from, effective_to) VALUES
-- Alice: had 2 versions (bronze -> gold)
(1, 'Alice Martin', 'bronze', '2024-01-01', '2024-06-01'),
(1, 'Alice Martin', 'gold',   '2024-06-01', '9999-12-31'),

-- Bob: only 1 version (always silver)
(2, 'Bob Singh',   'silver', '2024-01-01', '9999-12-31'),

-- Carol: had 2 versions (silver -> bronze, downgraded)
(3, 'Carol White', 'silver', '2024-01-01', '2024-09-01'),
(3, 'Carol White', 'bronze', '2024-09-01', '9999-12-31');


CREATE TEMP TABLE fct_order (
    order_id        SERIAL PRIMARY KEY,
    customer_id     INT,                  -- natural key (FK to dim)
    placed_at       TIMESTAMP,
    total_amount    NUMERIC(10,2)
);

INSERT INTO fct_order (customer_id, placed_at, total_amount) VALUES
-- Alice: 1 order before upgrade, 1 after
(1, '2024-03-15', 120.00),   -- should join to bronze version
(1, '2024-08-20', 340.00),   -- should join to gold version

-- Bob: 1 order (always silver)
(2, '2024-05-10', 89.50),    -- should join to silver version

-- Carol: 1 order before downgrade, 1 after
(3, '2024-07-04', 210.00),   -- should join to silver version
(3, '2024-11-11', 55.00);    -- should join to bronze version

Key things to notice:

* Alice appears twice with different customer_key and status — the time range correctly picks the right SCD2 version per order
* Carol similarly resolves to silver for her July order and bronze after her September downgrade
* No duplication of results — each order joins to exactly one dimension row

## Python for extracting, transforming, & loading data into a modelled destination

* We began by designing facts and dimensions from source tables. 

* Python is the system that executes the logic to take source data and turn it into your warehouse output.

* The concepts of ETL, ELT, and their variants describe the journey data takes from source to destination. 

* Python typically serves as the glue tying together three key parts

  - Extract: obtaining data from source systems and formats

  - Transform: applying business logic (like joins or enrichment) via processing systems 

  - Load: placing the transformed data into destination tables for stakeholder use.

#### Example

We will now build a snapshot dimension table for `dim_customer` built from the source tables `customer` and `customer_address`

In [ ]:
# We will use Spark to connect to source system (postgres), transform data with dataframes
# and load data into an iceberg table. So lets create a SparkSession to do this

from pyspark.sql import Row, SparkSession

spark = (
    SparkSession.builder.appName("dim_customer_snapshot")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# EXTRACT data from Upstream Source tables
# Connection parameters
url = "jdbc:postgresql://postgres:5432/ecommerce"
properties = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}

# Spark has a read.jdbc function to read directly from a database
customer_df = spark.read.jdbc(url=url, table="public.customer", properties=properties)
customer_address_df = spark.read.jdbc(
    url=url, table="public.customer_address", properties=properties
)

In [ ]:
# TRANSFORM upstream data into the dim_customer_snapshot compatable schema
# Transformation code to combine customer and customer_address into one table
customer_df.createOrReplaceTempView("customer")
customer_address_df.createOrReplaceTempView("customer_address") # 1:Many

transformed_df = spark.sql("""
    SELECT
        c.customer_id,
        c.email,
        c.full_name,
        c.phone,
        c.status,
        c.created_at,
        c.updated_at,
        COLLECT_LIST(
            STRUCT(
                ca.is_default,
                CONCAT(ca.line1, ', ', ca.city, ', ', ca.state, ', ', ca.country) AS address
            )
        ) AS addresses
    FROM customer c
    LEFT JOIN customer_address ca USING (customer_id)
    GROUP BY 1, 2, 3, 4, 5, 6, 7
""")
# we will cover list of struct in a later chapter

In [ ]:
# LOAD
# Write or Replace dim_customer_snapshot snapshot table
transformed_df.writeTo("local.warehouse.dim_customer_snapshot").createOrReplace()

**Note** 

`local.warehouse` schema was created as part of our Spark setup,


In [ ]:
# Check results
# We use .toPandas() for pretty display
spark.table("local.warehouse.dim_customer_snapshot").limit(2).toPandas()

#### Exercise [5 min]

Fill the below functions with the right code to create `dim_customer_snapshot`

In [ ]:
from pyspark.sql import DataFrame, SparkSession

url = "jdbc:postgresql://postgres:5432/ecommerce"
properties = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}


def extract(spark: SparkSession) -> dict[str, DataFrame]:
    pass


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    pass


def load(output_df: DataFrame) -> None:
    pass


def run(spark: SparkSession) -> None:
    # call the functions in the right order
    pass

In [ ]:
# This code should work
run(spark)

In [ ]:
# Your output should show up here
spark.table("local.warehouse.dim_customer_snapshot").limit(2).toPandas()

* We can see how the function names are representative of their operation.
* For dimensions, the transformation function joins multiple tables, ensuring each row represents a unique record at the desired level of detail.
* For fact tables, the transformation function will sanitize the data by removing invalid entries, standardize formats, and add calculated columns such as totals or ratios.
* The transform function is separate from the function to extract or load data into the destination.
* This separation lets us create simpler tests and keeps IO apart from transformations.
* IO is separate from the transformation to ensure orthogonality: changing the destination or input source requires updates only to those components, not to the transform function.

#### Exercise [10 min]

Build a pipeline for the fct_order_lines fact table using only the order_line table, and add the following columns.

1. is_discounted: computed as discount_amt > 0
2. effective_unit_price: unit_price - discount_amt

and only choose these columns for the output

[
“order_line_id”,
“order_id”,
“variant_id”,
“quantity”,
“unit_price”,
“discount_amt”,
“line_total”,
“created_at”,
“updated_at”,
“is_discounted”,
“effective_unit_price”,
]

In [ ]:
from pyspark.sql import DataFrame, SparkSession

JDBC_URL = "postgre_connection_url"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}


def extract(spark: SparkSession) -> dict[str, DataFrame]:
    pass


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    pass


def load(output_df: DataFrame) -> None:
    pass


def run(spark: SparkSession) -> None:
    # call the functions in the right order
    pass

In [ ]:
# This code should work
run(spark)

In [ ]:
# Your output should show up here
spark.table("local.warehouse.fct_order_lines").limit(2).toPandas()

## Data Storage Patterns: Partitioning for efficient reads & Bucketing for efficient joins/group by

* Modern data systems are designed to separate storage from compute, even when using vendor solutions such as Snowflake.
* Because storage and compute are separated, data processing engines can use compute power only when necessary.
* This flexibility, enabled by cheap storage and scalable compute, leads data processing vendors to charge by compute usage.
* Data storage patterns become critical, as they directly impact how quickly a query can run.
* The data storage pattern aims to minimize the amount of data read into the compute engine.
* To address this, two main data storage patterns emerge: `partitioning` and `bucketing`.

* **Partitioning** refers to storing data in folders, where each folder contains all data for a specific column value.


#### Example

Let’s create a partitioned version of the `fct_order_lines` table and inspect how the data is stored.

In [ ]:
# from pyspark.sql.functions import date
from pyspark.sql.functions.partitioning import days

spark.table("local.warehouse.fct_order_lines").writeTo(
    "local.warehouse.fct_order_lines_ct_partitioned"
).partitionedBy(days("created_at")).createOrReplace()

In [ ]:
spark.sql("describe extended local.warehouse.fct_order_lines_ct_partitioned").show()

In [ ]:
spark.sql(
    "select file_path from local.warehouse.fct_order_lines_ct_partitioned.files"
).limit(5).toPandas()



* Each created_at date has its own folder.
* If you query the table using `select * from local.warehouse.fct_order_lines_ct_partitioned where date(created_at) = '2025-04-23'`, only that single folder will be read. This significantly reduces the amount of data that would otherwise need to be filtered out.
* **Bucketing/clustering** is when the data is separated into n folders based on the hash of the column(s) used for the bucketing.

![Bucketing Example](images/lineitem_bucket.png)

* Bucketing is similar to partitioning, but instead of one partition per unique value of the column, you specify the number of buckets.
* Rows are stored in buckets (folders) based on the column(s) used for bucketing.
* Bucketing is helpful when you repeatedly group by/join by certain columns, as it ensures that the data processing engine (Spark, Snowflake, etc.) knows exactly where each value in the column is.
* **Cardinality** refers to the number of unique values of a column.
* Low-cardinality columns, such as date-time and entity codes (e.g., State/Country codes), are partitioned.
* High-cardinality columns, such as order timestamp, are bucketed.

#### Exercise [5 min]

* Assume that you have the below `group by` query run multiple times a day.
* What storage patterns will you apply to the `fct_order_lines` table to make the group by efficient?

```sql
select order_id
    , some_metrics
from fct_order_lines
group by order_id
```


**Rule of thumb**

**Partitioning:**

* Fact tables → partition by created time. Typically, by the creation date.
* Very large fact tables → partition by created hour
* Default to the coarsest granularity (minutes) that still gives you useful pruning. Make sure that the data per partition is not too small (< 500 MB)
* We partition by date, as most analytical queries will limit the time range (e.g., the past 2 years).
* Dimension/SCD2 → no partition (tables are small enough that partition overhead isn’t worth it). 
* Only when the dimension data is very large, consider bucketing by the join/group key (usually the ID) for SCD2 tables.

**Bucketing:**

* Only on columns that appear in frequent, repeated joins or group bys.
* Do not bucket speculatively; test to check that bucketing will meaningfully improve your performance.
* Typically, order_id, customer_id, and other join/group by keys for large fact tables, where the join is in the critical path of many downstream queries

The underlying principle across both:

* Storage optimization should be based on actual query patterns, not anticipated ones.
* Start with date partitioning, measure the impact, and add bucketing only if shuffle is proven to be the bottleneck.

**Note:** Deep dive into implementation details is in the [Spark for Data Engineers Course]().

#### Exercise [5 min]

Assume you decide to partition the fct_order_lines  table. Based on our Python pipeline above, where would you include this logic?

## Bus Matrix: Get everyone on the same page

* When designing your data model, always start with the business process to model
* A bus matrix is a process for breaking down warehouse deliverables into iterative chunks to minimize wasted effort.
* This makes it easier for people from different personas—such as engineering, Backend, analysts, DS, and bizops—to understand.

#### Example

* Let’s see how a simple bus matrix works.
* Create a matrix of business processes, grain, metrics, and the dimensions that they involve.
* For our business process of orders, payments, and shipments

| Business Process    | Grain                        | Metrics                                                             | dim_customer | dim_product_variant | dim_date | dim_warehouse | dim_return |
|---------------------|------------------------------|---------------------------------------------------------------------|:------------:|:-------------------:|:--------:|:-------------:|:----------:|
| Order / Sales       | 1 row per order_line         | revenue, quantity, discount_amt, line_total, avg_order_value        | Yes          | Yes                 | Yes      | No            | No         |
| Payments            | 1 row per payment            | payment_amount, payment_method, success_rate                        | Yes          | No                  | Yes      | No            | No         |
| Shipments           | 1 row per shipment_line      | shipped_qty, shipping_cost, days_to_ship, on_time_rate              | Yes          | Yes                 | Yes      | Yes           | No         |

* Build iteratively as business needs emerge

#### Exercise [5 min]

* Create a bus matrix for the  business process of Returns & user Sessions.
* **Hint**: Query the **returns and session** tables and propose potential metrics. Focus on learning the process, not on selecting exact metrics.

## Pipeline Types: Full refresh processes the entire source, and incremental processes a time range-specific source

* For example, when we created the `fct_order_lines` pipeline above, we processed the entire `order_line` input table on each run. This approach highlights a key difference between full and incremental processing methods.
* Reprocessing the entire input data is not always feasible, especially with fact tables
* This is where **incremental** data processing pattern is helpful.
* Incremental data processing pattern only processes a time range of source data with every run.
* The time range is typically provided as an input to the data processing script. 

* The responsibility of generating the right start and end time is on the scheduling system (which we will cover in a later chapter)


#### Example

Now, we will examine how the `fct_order_lines` pipeline can be modified to process data incrementally, as the `fct_order_lines_incremental` pipeline.

In [ ]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

TABLE_NAME = "local.warehouse.fct_order_lines_incremental"

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}


def extract(
    spark: SparkSession,
    start_time: str,
    end_time: str,
) -> dict[str, DataFrame]:
    order_line_df = spark.read.jdbc(
        url=JDBC_URL,
        table="public.order_line",
        properties=JDBC_PROPERTIES,
    )
    return {
        "order_line": order_line_df.filter(
            (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
        )
    }


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    return input_dfs["order_line"].select(
        F.col("order_line_id"),
        F.col("order_id"),
        F.col("variant_id"),
        F.col("quantity"),
        F.col("unit_price"),
        F.col("discount_amt"),
        F.col("line_total"),
        F.col("created_at"),
        F.col("updated_at"),
        (F.col("discount_amt") > 0).alias("is_discounted"),
        (F.col("unit_price") - F.col("discount_amt")).alias("effective_unit_price"),
    )


def load(output_df: DataFrame, spark: SparkSession) -> None:
    if not spark.catalog.tableExists(TABLE_NAME):
        (
            output_df.writeTo(TABLE_NAME)
            .partitionedBy(F.partitioning.days("created_at"))
            .createOrReplace()
        )
    else:
        output_df.writeTo(TABLE_NAME).overwritePartitions()


def run(spark: SparkSession, start_time: str, end_time: str) -> None:
    load(transform(extract(spark, start_time, end_time)), spark)

In [ ]:
run(spark, "2025-01-01", "2026-01-01")

In [ ]:
spark.table("local.warehouse.fct_order_lines_incremental").limit(3).toPandas()

In [ ]:
# check min and max dates to confirm with the start and end time input args
spark.sql("""
select min(created_at) as min_created_at
, max(created_at) as max_created_at
from local.warehouse.fct_order_lines_incremental
""").limit(1).toPandas()

* We can see how in the extract function we filter for the time range based on the created_at column.

```python
return {
        "order_line": order_line_df.filter(
            (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
        )
    }
```

* Note that we use created_at for filtering since the source table is append-only.
* Also note that the filter includes the start time but excludes the end time. This is because when we run incremental pipelines (e.g., at a yearly frequency), we want to capture all the data only from the last year. This will translate to the range.

```sql
created_at >= 2025-01-01 00:00:00 and created_at < 2026-01-01 00:00:00
```

* This filter ensures we capture only 2025 events, and avoid events at 2026-01-01 00:00:00.
* The transform function remains the same. Let’s look at the load function.
```python
if not spark.catalog.tableExists(TABLE_NAME):
        (
            output_df.writeTo(TABLE_NAME)
            .partitionedBy(F.partitioning.days("created_at"))
            .createOrReplace()
        )
else:
    output_df.writeTo(TABLE_NAME).overwritePartitions()
```
* We create a table if it does not exist and overwrite the partitions if it exists.
* We will see how overwritePartitions leads to easy-to-maintain pipelines in a later chapter. But the key idea is that, no matter how many times we re-run the pipeline, the output will reflect the entire input.
* Fact tables are typically implemented as incremental pipelines due to the size of the data. 
* Dimension table pipelines typically re-process the entire source data.

#### Exercise [20 min]

For this exercise, create an incremental version of fct_orders based on the upstream order & payment tables.

Here is the logic for the transformation.

```sql
SELECT
    o.order_id,
    o.customer_id,
    o.shipping_addr_id,
    o.billing_addr_id,
    o.status                                    AS order_status,
    o.total_amount,
    o.placed_at,
    p.payment_id,
    p.method                                    AS payment_method,
    p.amount                                    AS payment_amount,
    p.status                                    AS payment_status,
    p.provider_ref,
    p.paid_at,
    o.created_at,
    o.updated_at,
    (p.status = 'completed')                    AS is_paid,
    DATEDIFF(p.paid_at, o.placed_at)            AS days_to_payment
FROM order o
LEFT JOIN payment p USING (order_id)
```

In [ ]:
run(spark, "2025-01-01", "2026-01-01")

In [ ]:
spark.table("local.warehouse.fct_orders").limit(3).toPandas()

In [ ]:
# check min and max dates to confirm with the start and end time input args
spark.sql("""
select min(created_at) as min_created_at
, max(created_at) as max_created_at
from local.warehouse.fct_orders
""").limit(1).toPandas()

* Snapshot pipelines are simpler compared to incremental pipelines

## Recap

In this section, we covered the key fundamentals of 

1. Data Modeling with Kimball
2. Creating data pipelines
3. Deep dive into fact and dimension types
4. Differences between incremental and full-refresh pipelines
5. Data storage patterns to optimize for analytical usage

In the next section, we will build on these to extend our pipelines and improve their fault resistance.